# E5 — Régularisation de Tikhonov
**Chapitre 6 — Expériences numériques et calibration**

Cet notebook implémente la régularisation de Tikhonov (§ 5.3) :
$$L_\alpha(\lambda,\delta) = \frac{1}{N}\|F(\lambda,\delta)\|^2 + \alpha\,\|(\lambda,\delta)-(\lambda_0,\delta_0)\|^2$$

Expériences :
- **E5a** — L-curve : choix de α par la méthode de la L-curve (§ 5.3.2)
- **E5b** — Calibration avec α optimal sur données bruitées
- **E5c** — Effet de α sur la stabilité temporelle des paramètres

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join('..', 'src'))
os.makedirs('../figures', exist_ok=True)

from merton_calib.calibration import (
    synthetic_smile, calibrate_tikhonov, lcurve, lcurve_kink
)

plt.rcParams.update({'font.family': 'DejaVu Serif', 'font.size': 11,
                     'axes.grid': True, 'grid.alpha': 0.3, 'figure.dpi': 120})

# ── Paramètres de référence ───────────────────────────────────────────────
S0, r         = 100.0, 0.05
SIGMA0, MUJ0  = 0.20, -0.10
LAM_TRUE      = 1.0
DELTA_TRUE    = 0.15
LAM0, DELTA0  = 0.8, 0.12     # prior a priori θ₀

STRIKES    = np.tile([80., 90., 100., 110., 120.], 4)
MATURITIES = np.repeat([0.25, 0.5, 1.0, 2.0], 5)
N_OPT      = len(STRIKES)

NOISE_STD  = 0.005   # 0.5 vol-pt de bruit (équivalent demi bid-ask)
sigma_mkt_clean = synthetic_smile(S0, r, SIGMA0, MUJ0, LAM_TRUE, DELTA_TRUE,
                                    STRIKES, MATURITIES, noise_std=0.0)
sigma_mkt_noisy = synthetic_smile(S0, r, SIGMA0, MUJ0, LAM_TRUE, DELTA_TRUE,
                                    STRIKES, MATURITIES, noise_std=NOISE_STD, seed=1)
print(f'Données bruitées : RMSE bruit = {np.std(sigma_mkt_noisy - sigma_mkt_clean)*100:.3f} vol-pts')

## E5a — L-curve : choix du paramètre α

In [ ]:
alpha_grid = np.logspace(-6, 0, 25)

lc_clean = lcurve(
    sigma_mkt_clean, STRIKES, MATURITIES, S0, r, SIGMA0, MUJ0,
    LAM0, DELTA0, method='tikhonov', reg_params=alpha_grid,
    n_starts=5, seed=42
)
lc_noisy = lcurve(
    sigma_mkt_noisy, STRIKES, MATURITIES, S0, r, SIGMA0, MUJ0,
    LAM0, DELTA0, method='tikhonov', reg_params=alpha_grid,
    n_starts=5, seed=42
)
print('L-curve calculée.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# ── L-curve données sans bruit ────────────────────────────────────────────
ax = axes[0]
rmse_c  = lc_clean['RMSE'] * 100
norm_c  = np.sqrt(lc_clean['reg_norm'])
ax.loglog(rmse_c, norm_c, 'b.-', ms=7, lw=1.5)
for k, a in enumerate(alpha_grid[::4]):
    ax.annotate(f'{a:.0e}',
                (rmse_c[k*4], norm_c[k*4]),
                fontsize=6.5, textcoords='offset points', xytext=(4,2))
ax.set_xlabel('RMSE de calibration (vol-pts)'); ax.set_ylabel(r'$\|\theta^*-\theta_0\|$')
ax.set_title('L-curve Tikhonov — données sans bruit')

# ── L-curve données bruitées ──────────────────────────────────────────────
ax = axes[1]
rmse_n  = lc_noisy['RMSE'] * 100
norm_n  = np.sqrt(lc_noisy['reg_norm'])
ax.loglog(rmse_n, norm_n, 'r.-', ms=7, lw=1.5)
# Identifier le coude (max courbure approx : point le plus proche de l'angle)
kink_idx = lcurve_kink(lc_noisy['RMSE'], lc_noisy['reg_norm'])
ax.plot(rmse_n[kink_idx], norm_n[kink_idx], 'k^', ms=10,
        label=fr'$\alpha^*$ ≈ {alpha_grid[kink_idx]:.1e}', zorder=5)
ax.set_xlabel('RMSE de calibration (vol-pts)'); ax.set_ylabel(r'$\|\theta^*-\theta_0\|$')
ax.set_title('L-curve Tikhonov — données bruitées (σ=0.5 vol-pt)')
ax.legend()

plt.tight_layout()
plt.savefig('../figures/E5_lcurve_tikhonov.pdf', bbox_inches='tight')
plt.show()

alpha_opt = float(alpha_grid[kink_idx])
print(f'α optimal (coude) : {alpha_opt:.2e}')

## E5b — Calibration avec α optimal

In [ ]:
# Calibration avec α=0 et α=α_opt, données bruitées
res_noreg = calibrate_tikhonov(
    sigma_mkt_noisy, STRIKES, MATURITIES, S0, r, SIGMA0, MUJ0,
    alpha=0.0, lam0=LAM0, delta0=DELTA0, n_starts=10, seed=42
)
res_tik = calibrate_tikhonov(
    sigma_mkt_noisy, STRIKES, MATURITIES, S0, r, SIGMA0, MUJ0,
    alpha=alpha_opt, lam0=LAM0, delta0=DELTA0, n_starts=10, seed=42
)

print('Résultats sur données bruitées (bruit = 0.5 vol-pt) :')
print(f'          λ_vrai={LAM_TRUE:.3f}  δ_vrai={DELTA_TRUE:.3f}')
print(f'α = 0   : λ*={res_noreg["lambda_"]:.3f}  δ*={res_noreg["delta"]:.3f}  RMSE={res_noreg["RMSE"]*100:.3f}')
print(f'α = α*  : λ*={res_tik["lambda_"]:.3f}  δ*={res_tik["delta"]:.3f}  RMSE={res_tik["RMSE"]*100:.3f}')

In [ ]:
# Visualisation du smile reconstitué
from merton_calib.implied_vol import merton_implied_vol

T_vals = np.unique(MATURITIES)
K_vals = np.unique(STRIKES)

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
axes = axes.flatten()
colors = ['#1f77b4','#ff7f0e','#2ca02c','#d62728']

for j, T in enumerate(T_vals):
    ax = axes[j]
    mask = MATURITIES == T
    K_sub = STRIKES[mask]

    sm_true  = sigma_mkt_clean[mask]*100
    sm_noisy = sigma_mkt_noisy[mask]*100

    # Smile reconstitué
    def smile_model(lam, delta, K_arr):
        return np.array([merton_implied_vol(S0, k, T, r, SIGMA0, lam, MUJ0, delta)*100
                         for k in K_arr])

    sm_noreg = smile_model(res_noreg['lambda_'], res_noreg['delta'], K_sub)
    sm_tik   = smile_model(res_tik['lambda_'],   res_tik['delta'],   K_sub)

    ax.plot(K_sub, sm_true,  'k-',  lw=2,  label='Vrai')
    ax.plot(K_sub, sm_noisy, 'ko',  ms=6,  label='Marché bruité')
    ax.plot(K_sub, sm_noreg, 'b--', lw=1.5, label='α=0')
    ax.plot(K_sub, sm_tik,   'r-',  lw=1.5, label=f'α=α*={alpha_opt:.1e}')
    ax.set_title(f'T = {T} an')
    ax.set_xlabel('Strike K'); ax.set_ylabel(r'$\sigma_{\rm imp}$ (%)')
    if j == 0: ax.legend(fontsize=8)

plt.suptitle('Smile reconstitué — Tikhonov vs. sans régularisation', y=1.02)
plt.tight_layout()
plt.savefig('../figures/E5_smile_tikhonov.pdf', bbox_inches='tight')
plt.show()

## E5c — Stabilité temporelle des paramètres calibrés

In [ ]:
# Simuler 30 jours de données de marché avec évolution lente des vrais params
n_days      = 30
lam_path    = LAM_TRUE   + 0.002 * np.arange(n_days)    # légère dérive
delta_path  = DELTA_TRUE + 0.001 * np.arange(n_days)

rng = np.random.default_rng(99)
lam_noreg, lam_tik   = [], []
delta_noreg, delta_tik = [], []

for day in range(n_days):
    sm = synthetic_smile(S0, r, SIGMA0, MUJ0, lam_path[day], delta_path[day],
                          STRIKES, MATURITIES,
                          noise_std=NOISE_STD, seed=int(rng.integers(1e6)))
    r0 = calibrate_tikhonov(sm, STRIKES, MATURITIES, S0, r, SIGMA0, MUJ0,
                              alpha=0.0, lam0=LAM0, delta0=DELTA0,
                              n_starts=5, seed=42)
    r1 = calibrate_tikhonov(sm, STRIKES, MATURITIES, S0, r, SIGMA0, MUJ0,
                              alpha=alpha_opt, lam0=LAM0, delta0=DELTA0,
                              n_starts=5, seed=42)
    lam_noreg.append(r0['lambda_']);  delta_noreg.append(r0['delta'])
    lam_tik.append(r1['lambda_']);    delta_tik.append(r1['delta'])

days = np.arange(n_days)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(days, lam_path, 'k-', lw=2, label='λ vrai')
axes[0].plot(days, lam_noreg, 'b.--', ms=5, lw=0.8, alpha=0.7, label='α=0')
axes[0].plot(days, lam_tik,   'r.-',  ms=5, lw=1.2, label=f'Tikhonov α*')
axes[0].set_ylabel(r'$\lambda^*$'); axes[0].legend(fontsize=9)
axes[0].set_title('Stabilité temporelle des paramètres calibrés')

axes[1].plot(days, delta_path, 'k-', lw=2, label='δ vrai')
axes[1].plot(days, delta_noreg, 'b.--', ms=5, lw=0.8, alpha=0.7, label='α=0')
axes[1].plot(days, delta_tik,   'r.-',  ms=5, lw=1.2, label=f'Tikhonov α*')
axes[1].set_ylabel(r'$\delta^*$'); axes[1].set_xlabel('Jour')
axes[1].legend(fontsize=9)

# Statistiques
print(f'Variance temporelle λ* :')
print(f'  α=0   : std={np.std(lam_noreg):.4f}')
print(f'  Tikh. : std={np.std(lam_tik):.4f}  (réduction ×{np.std(lam_noreg)/max(np.std(lam_tik),1e-8):.1f})')

plt.tight_layout()
plt.savefig('../figures/E5_stabilite_temporelle.pdf', bbox_inches='tight')
plt.show()

**Conclusion E5.** La régularisation de Tikhonov améliore significativement la stabilité temporelle
des paramètres calibrés. Le coude de la L-curve identifie un α optimal qui équilibre
fidélité aux données et stabilité, conformément au principe de Morozov (§ 5.3.2).